# NLBSE'27 Code Classification: early-exit UniXcoder (training)

**One multi-label model** predicting `[Vulnerable, MAT]` from `Function`, built for the official score

$$\text{score} = 0.6\,\overline{F_1} + 0.2\max\!\Big(\tfrac{1000-\text{runtime}}{1000},0\Big) + 0.2\max\!\Big(\tfrac{3\,000\,000-\text{GFLOPs}}{3\,000\,000},0\Big)$$

The CodeBERT baseline scores **0.7299** (F1 0.910, 192 s, 2.67M GFLOPs on a Colab T4). The GFLOPs cap is barely
above the baseline, so compute is worth as much as accuracy. The method:

| Component | What | Why |
|---|---|---|
| Teacher | UniXcoder-base-nine (12 layers) fine-tuned by `configs/T0_teacher12.json` (4-GPU DDP) | Strongest model we can distill from; reproducible from this repo |
| Depth pruning | Student = selected teacher layers (e.g. 2, 5, 8, 11) | Keeps pretrained + task knowledge |
| Width pruning (optional) | Keep the most important FFN neurons (activation x weight norm) | FFN is ~2/3 of the FLOPs |
| Whitespace compaction | Collapse formatting whitespace | -16% tokens, truncation 3.8% to 0.5% |
| SATD flag tokens | `<satd_c>` if `TODO/FIXME/HACK/XXX` is in a comment, `<satd_w>` if elsewhere | Computed on the full text, so truncation cannot hide it |
| Distillation | BCE on labels + temperature-scaled BCE on teacher logits, at every exit | Transfers the teacher's soft knowledge |
| Early exit | One head per layer; confident rows leave the batch | Easy rows cost 1-2 layers instead of 4 |
| Score-aware calibration | Per-exit thresholds + margins chosen to maximize the official score | Optimizes what is ranked |
| Options (stage A2) | `ema_decay`, `exit0` + `head: attn`, `teacher_logits: oof` (out-of-fold soft targets from `configs/T0_fold*.json`) | EMA weights; an exit before the first layer; distillation that is not a copy of the labels |

**Data use.** With `validation: inner` (default) a stratified 8% split of the training set selects checkpoints,
thresholds, exit margins and the winning configuration; the public test split is measured once per model at
the end. The submission recipe (`scripts/run_stage_b.sh`, `validation: none`) retrains the winner on all training
rows for its selected number of epochs and calibrates thresholds on the public test, which the guide designates
for tuning. Only `Function` is used as input; one model predicts both labels.


## 1. Configuration
Train the teacher first (`bash scripts/run_stage_a.sh` does it, or `python -m torch.distributed.run --nproc_per_node 4 -m nlbse27ee.train --config configs/T0_teacher12.json`). Then pick a student config here. `quick > 0` subsamples for a smoke test.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "nlbse27ee").exists() else Path.cwd().parent   # repo root
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

CONFIG = "configs/R8_6L_ffn1536_exits.json"
OVERRIDES = dict(
    teacher_checkpoint="teacher/unixcoder12_teacher.pt",
    gpu=0,
    quick=0,
)

In [ ]:
import numpy as np, torch
from nlbse27ee import train as T, exits as ex
from nlbse27ee.evaluate import report_table
from nlbse27ee.preprocess import compact, satd_flag

cfg = T.TrainConfig.from_json(CONFIG, **OVERRIDES)
print(json.dumps(T.asdict(cfg), indent=2))

## 2. Data
With `validation: inner`: 197,419 rows train, 17,167 rows (stratified, seed 42) select, and the 30,655-row public test is measured once at the end. The hidden test belongs to the organizers.

In [ ]:
ctx = T.setup(cfg)
print("split", ctx["split_sha256"])
for name, labels in (("train", ctx["train_labels"]), ("validation", ctx["val_labels"]),
                     ("public test", ctx["public_labels"])):
    print(f"{name:>10}: n={len(labels):,}  vulnerable={labels[:, 0].mean():.4f}  MAT={labels[:, 1].mean():.4f}")

## 3. Preprocessing: compaction and SATD flags
Flags are computed on the **full** text before truncation. Their precision/recall against the MAT label on the training set is shown below.

In [ ]:
sample = ctx["train"][3]["Function"]
print(repr(sample[:300])); print("->", repr(compact(sample, cfg.compact_mode)[:300]))

flags = np.array([satd_flag(t or "") for t in ctx["train"]["Function"]], dtype=object)
mat = ctx["train_labels"][:, 1] == 1
for flag in ("<satd_c>", "<satd_w>"):
    hit = flags == flag
    print(f"{flag}: rows {hit.sum():,}, precision {mat[hit].mean():.4f}, recall {(hit & mat).sum() / mat.sum():.4f}")
print(f"MAT positives without any tag: {(mat & (flags == None)).sum() / mat.sum():.1%} (not recoverable from text)")

In [ ]:
T.encode_inputs(ctx)
for key in ("teacher_train_ids", "train_ids", "val_ids"):
    lengths = ctx[key].lengths
    print(f"{key:>18}: mean tokens {lengths.mean():6.1f}  p99 {np.percentile(lengths, 99):.0f}  at 512: {(lengths >= 512).mean():.4f}")

## 4. Our encoder equals `transformers`' RobertaModel
The student uses a compact re-implementation (fused QKV, SDPA attention, early exit). This check loads the full teacher into both and compares logits.

In [ ]:
from nlbse27ee.model import load_teacher_state, verify_against_transformers
state = load_teacher_state(cfg.teacher_checkpoint)
print("max |logit difference| vs transformers:",
      verify_against_transformers(state, cfg.tokenizer_id, cfg.tokenizer_revision, ctx["device"]))
del state

## 5. Teacher targets
Teacher logits on the training rows, in the teacher's input format (by default the same compaction and flags as the student), cached by teacher SHA-256 and split; or out-of-fold logits from fold teachers when `teacher_logits` is `oof`.

In [ ]:
T.teacher_targets(ctx)
agree = ((ctx["teacher_logits"] > 0) == (ctx["train_labels"] == 1)).mean(0)
print("teacher sign agreement with labels [vuln, MAT]:", agree.round(4))

## 6. Student
Selected teacher layers, optionally FFN-pruned; every exit head starts from the teacher classifier; flag-token embeddings start from the mean of the tag tokens.

In [ ]:
model = T.build_student(ctx)

## 7. Training
Loss = sum over exits of $w_k\,[(1-\alpha)\,\text{BCE}(y) + \alpha T^2\,\text{BCE}(\sigma(t/T))]$ with $w_k \propto k$. Length-bucketed batches, fp16 AMP, AdamW, linear warmup/decay, early stopping on final-exit validation mean F1. Re-running resumes from `last.pt`.

In [ ]:
training = T.fit(ctx, model)
print("best epoch", training["best_epoch"], "val mean F1", round(training["best_f1"], 4))

In [ ]:
import matplotlib.pyplot as plt
history = training["history"]
curves = np.array([row["val_mean_f1_per_exit"] for row in history])
for k in range(curves.shape[1]):
    plt.plot([row["epoch"] for row in history], curves[:, k], marker="o", label=f"exit {k + 1}")
plt.xlabel("epoch"); plt.ylabel("validation mean F1"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 8. Calibration and official-protocol measurement
1. Per-exit F1-optimal thresholds.
2. The organizers' baseline cell is run once on this GPU to get a V100 to T4 runtime factor (cached).
3. Exit margins are searched to maximize the official score (runtime modeled as proportional to GFLOPs).
4. A 2-fold holdout shows how optimistic the tuning is.
5. The final model is measured with the organizers' exact loop (batch 600, fp16 autocast, profiler, 10 repetitions); batches are formed after sorting by length.

In [ ]:
calibration = T.calibrate(ctx, model)
print("per-exit validation mean F1:", np.round(calibration["per_exit_val_mean_f1"], 4))
print("FLOPs simulator / profiler:", round(calibration["flops_simulator_ratio"], 4))
if "holdout" in calibration:
    print("holdout check:", json.dumps(calibration["holdout"], indent=1))

## 9. Results (paper table)

In [ ]:
final = calibration["final"]
print(report_table(final))
print()
print(f"GFLOPs (profiler, whole set): {final['gflops']:,.0f}")
print(f"Runtime: {final['local_runtime_s']:.2f} s local ({final['gpu']}) -> estimated T4 {final['estimated_t4_runtime_s']:.1f} s")
print(f"Exits per layer: {final['exit_histogram']}")
print(f"Public test held out: {final['held_out']}")
print(f"Estimated official score on the public test: {final['estimated_t4_score']:.4f}   (baseline 0.7299)")
print(f"Selection split: F1 {calibration['selection']['mean_f1']:.4f}, estimated score {calibration['selection']['estimated_score']:.4f}"
      f" (without early exit {calibration['selection_no_exit']['estimated_score_at_public_scale']:.4f})")

## 10. Export
A self-contained folder: fp16 weights, tokenizer, exit policy, and the `nlbse27ee` code. Upload it to the team HF repo or Google Drive and run `NLBSE27_Submission_Colab_T4.ipynb`.

In [ ]:
export_dir = T.export(ctx, model, calibration, training)
print(sorted(p.name for p in export_dir.iterdir()))

## 11. All runs
After `scripts/run_stage_a.sh` / `run_stage_a2.sh`, runs are ranked by the score estimated on the selection split; public-test columns are the one-time measurement.

In [ ]:
from nlbse27ee.summarize import collect, markdown
from IPython.display import Markdown
Markdown(markdown(collect(cfg.work_root)))